# 10 - Exploratory Data Analysis

This notebook analyzes the final analytical master tables and exports summary tables to `outputs/tables/`.

The analysis is organized around the project theme: **Cameroonian International Migration Trends (2015-2024): Destinations, Entry Reasons and Post-Arrival Trajectories Across Available Reference Years**.

The goal is to answer the three portfolio research questions while keeping indicators methodologically separated. 


In [65]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


In [66]:
PROJECT_ROOT = Path('..')

ANALYTICAL_PATH = PROJECT_ROOT / 'data' / 'processed' / 'analytical'
OUTPUTS_PATH = PROJECT_ROOT / 'outputs'
TABLES_PATH = OUTPUTS_PATH / 'tables'
FIGURES_PATH = OUTPUTS_PATH / 'figures'

TABLES_PATH.mkdir(parents=True, exist_ok=True)
FIGURES_PATH.mkdir(parents=True, exist_ok=True)

period_order = ["2015-2019", "2020-2021", "2022-2024"]


## Load Analytical Tables

The notebook uses the three master tables created in notebook 09.


In [67]:
destinations = pd.read_csv(ANALYTICAL_PATH / 'destinations_master.csv')
entry_reasons = pd.read_csv(ANALYTICAL_PATH / 'entry_reasons_master.csv')
post_arrival = pd.read_csv(ANALYTICAL_PATH / 'post_arrival_master.csv')


In [68]:
def prepare_eda_table(df):
    """Convert analytical columns to stable types before aggregation."""
    df = df.copy()

    if 'year' in df.columns:
        df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')

    if 'value' in df.columns:
        df['value'] = pd.to_numeric(df['value'], errors='coerce').fillna(0)

    text_columns = [
        'source',
        'dataset',
        'origin_country',
        'destination_country',
        'period',
        'measure_type',
        'reason',
        'sub_reason',
        'analysis_question',
    ]

    for col in text_columns:
        if col in df.columns:
            df[col] = df[col].astype('string').str.strip()

    return df


def assign_period(year):
    """Assign neutral descriptive reference periods based on the observation year."""
    if pd.isna(year):
        return 'outside_scope'

    year = int(year)

    if 2015 <= year <= 2019:
        return '2015-2019'
    if 2020 <= year <= 2021:
        return '2020-2021'
    if 2022 <= year <= 2024:
        return '2022-2024'

    return 'outside_scope'


def export_csv(df, file_name):
    """Export a table to the portfolio output folder."""
    df.to_csv(TABLES_PATH / file_name, index=False)
    return df


In [69]:
destinations = prepare_eda_table(destinations)
entry_reasons = prepare_eda_table(entry_reasons)
post_arrival = prepare_eda_table(post_arrival)

# Recode the existing `period` column with neutral reference periods.
# This keeps the previous variable name while removing the previous event-centered framing.
entry_reasons['period'] = entry_reasons['year'].apply(assign_period)
post_arrival['period'] = post_arrival['year'].apply(assign_period)

entry_reasons = entry_reasons[entry_reasons['period'] != 'outside_scope'].copy()
post_arrival = post_arrival[post_arrival['period'] != 'outside_scope'].copy()

entry_reasons['period'] = pd.Categorical(
    entry_reasons['period'],
    categories=period_order,
    ordered=True,
)

post_arrival['period'] = pd.Categorical(
    post_arrival['period'],
    categories=period_order,
    ordered=True,
)

for name, df in {
    'destinations': destinations,
    'entry_reasons': entry_reasons,
    'post_arrival': post_arrival,
}.items():
    print()
    print(name.upper())
    print('Shape:', df.shape)
    print('Columns:', df.columns.tolist())
    if 'source' in df.columns:
        print('Sources:', sorted(df['source'].dropna().unique()))
    if 'measure_type' in df.columns:
        print('Measure types:', sorted(df['measure_type'].dropna().unique()))
    if 'period' in df.columns:
        print('Periods:', list(df['period'].dropna().unique()))



DESTINATIONS
Shape: (695, 11)
Columns: ['source', 'dataset', 'origin_country', 'destination_country', 'year', 'measure_type', 'value', 'total_migrants', 'male_migrants', 'female_migrants', 'analysis_question']
Sources: ['oecd', 'undesa']
Measure types: ['inflow', 'stock', 'stock_birth', 'stock_nationality']

ENTRY_REASONS
Shape: (1595, 11)
Columns: ['source', 'dataset', 'origin_country', 'destination_country', 'year', 'reason', 'sub_reason', 'measure_type', 'value', 'analysis_question', 'period']
Sources: ['eurostat', 'oecd']
Measure types: ['asylum_inflow', 'permit', 'seasonal_worker_inflow', 'worker_inflow']
Periods: ['2015-2019', '2020-2021', '2022-2024']

POST_ARRIVAL
Shape: (948, 9)
Columns: ['source', 'dataset', 'origin_country', 'destination_country', 'year', 'measure_type', 'value', 'analysis_question', 'period']
Sources: ['eurostat', 'oecd']
Measure types: ['citizenship_acquisition', 'long_term_resident', 'status_change']
Periods: ['2015-2019', '2020-2021', '2022-2024']


## Q1 - Destination Countries

UN DESA stock data is the main source for Q1 because it provides the most consistent global view of Cameroonian migrant stocks by destination country. OECD indicators are exported separately as complementary evidence.


In [70]:
undesa_destinations = destinations[
    (destinations['source'].str.lower() == 'undesa')
    & (destinations['measure_type'].str.lower() == 'stock')
].copy()

top_destinations_by_year = {}
for year in [2015, 2020, 2024]:
    top_table = (
        undesa_destinations[undesa_destinations['year'] == year]
        .groupby('destination_country', as_index=False)['value']
        .sum()
        .sort_values('value', ascending=False)
        .head(10)
    )
    top_destinations_by_year[year] = export_csv(top_table, f'top_10_destinations_{year}.csv')

top_destinations_by_year[2024]


,destination_country,value
18,France,127995.0
37,Nigeria,56288.0
19,Gabon,54949.0
5,Canada,43662.0
6,Chad,40472.0
1,Belgium,30634.0
26,Italy,16244.0
9,Congo,11167.0
43,South Africa,4916.0
32,Mali,2339.0


In [71]:
destination_evolution = (
    undesa_destinations[undesa_destinations['year'].isin([2015, 2020, 2024])]
    .pivot_table(
        index='destination_country',
        columns='year',
        values='value',
        aggfunc='sum',
    )
    .reset_index()
)

for year in [2015, 2020, 2024]:
    if year not in destination_evolution.columns:
        destination_evolution[year] = pd.NA

destination_evolution['change_2015_2024'] = destination_evolution[2024] - destination_evolution[2015]
destination_evolution['pct_change_2015_2024'] = (
    destination_evolution['change_2015_2024'] / destination_evolution[2015] * 100
)

top_increases = export_csv(
    destination_evolution.dropna(subset=[2015, 2024])
    .sort_values('change_2015_2024', ascending=False)
    .head(10),
    'top_destination_increases_2015_2024.csv',
)

top_decreases = export_csv(
    destination_evolution.dropna(subset=[2015, 2024])
    .sort_values('change_2015_2024', ascending=True)
    .head(10),
    'top_destination_decreases_2015_2024.csv',
)

top_increases


year,destination_country,2015,2020,2024,change_2015_2024,pct_change_2015_2024
18,France,97288.0,114528.0,127995.0,30707.0,31.562988
37,Nigeria,26893.0,52489.0,56288.0,29395.0,109.303536
5,Canada,18062.0,27172.0,43662.0,25600.0,141.734027
1,Belgium,16560.0,22683.0,30634.0,14074.0,84.987923
6,Chad,30702.0,35635.0,40472.0,9770.0,31.822031
19,Gabon,46269.0,50906.0,54949.0,8680.0,18.759861
26,Italy,10856.0,15250.0,16244.0,5388.0,49.631540
11,Cyprus,54.0,1885.0,2001.0,1947.0,3605.555556
31,Luxembourg,59.0,1234.0,1425.0,1366.0,2315.254237
21,Greece,460.0,1545.0,1641.0,1181.0,256.739130


In [72]:
oecd_destination_summary = (
    destinations[destinations['source'].str.lower() == 'oecd']
    .groupby(['measure_type', 'year'], as_index=False)['value']
    .sum()
    .sort_values(['measure_type', 'year'])
)

export_csv(oecd_destination_summary, 'oecd_destination_indicators_summary.csv')


,measure_type,year,value
0,inflow,2015,16826.0
1,inflow,2016,18264.0
2,inflow,2017,19071.0
3,inflow,2018,19276.0
4,inflow,2019,20540.0
5,inflow,2020,13529.0
6,inflow,2021,18161.0
7,inflow,2022,3839.0
8,stock_birth,2015,214799.0
9,stock_birth,2016,198182.0


## Q2 - Observable Reasons for Entry

Eurostat first permits are the main source for Q2. They are interpreted as administrative entry indicators, not as migrant stock data.

OECD worker inflow, seasonal worker inflow and asylum inflow indicators are exported separately as complementary evidence only.

The `period` variable is used as a descriptive reference grouping:

- `2015-2019`
- `2020-2021`
- `2022-2024`

These periods are used to summarize available years. They are not used to estimate causal effects.


In [73]:
eurostat_entry = entry_reasons[
    (entry_reasons['source'].str.lower() == 'eurostat')
    & (entry_reasons['measure_type'].str.lower() == 'permit')
].copy()

oecd_entry = entry_reasons[
    entry_reasons['source'].str.lower() == 'oecd'
].copy()

# Keep the existing `period` variable and ensure that it uses neutral labels.
eurostat_entry['period'] = eurostat_entry['year'].apply(assign_period)
eurostat_entry = eurostat_entry[eurostat_entry['period'] != 'outside_scope'].copy()

eurostat_entry['period'] = pd.Categorical(
    eurostat_entry['period'],
    categories=period_order,
    ordered=True,
)

reason_totals = export_csv(
    eurostat_entry
    .groupby('reason', as_index=False)['value']
    .sum()
    .sort_values('value', ascending=False),
    'entry_reasons_total_by_reason.csv',
)

reason_yearly = export_csv(
    eurostat_entry
    .groupby(['year', 'reason'], as_index=False)['value']
    .sum()
    .sort_values(['year', 'reason']),
    'entry_reasons_yearly_evolution.csv',
)

reason_by_period = (
    eurostat_entry
    .groupby(['period', 'reason'], as_index=False, observed=False)['value']
    .sum()
)

reason_by_period['period'] = pd.Categorical(
    reason_by_period['period'],
    categories=period_order,
    ordered=True,
)

reason_by_period = export_csv(
    reason_by_period
    .sort_values(['period', 'reason']),
    'entry_reasons_by_period.csv',
)

period_totals = (
    reason_by_period
    .groupby('period', as_index=False, observed=False)['value']
    .sum()
    .rename(columns={'value': 'period_total'})
)

reason_share_by_period = reason_by_period.merge(
    period_totals,
    on='period',
    how='left',
)

reason_share_by_period['share_pct'] = (
    reason_share_by_period['value']
    / reason_share_by_period['period_total']
    * 100
)

reason_share_by_period = export_csv(
    reason_share_by_period,
    'entry_reasons_share_by_period.csv',
)

dominant_entry_reason_by_period = export_csv(
    reason_share_by_period
    .sort_values(['period', 'share_pct'], ascending=[True, False])
    .groupby('period', observed=False)
    .head(1)
    .reset_index(drop=True),
    'dominant_entry_reason_by_period.csv',
)

reason_totals


,reason,value
0,education,56458.0
1,family,55425.0
2,other,30443.0
3,work,10681.0


In [74]:
# OECD is kept separate from Eurostat first permits.
# These indicators are exported as complementary evidence only.

oecd_entry_summary = (
    oecd_entry
    .groupby(['measure_type', 'sub_reason', 'year'], as_index=False)['value']
    .sum()
    .sort_values(['measure_type', 'sub_reason', 'year'])
)

export_csv(oecd_entry_summary, 'oecd_entry_indicators_summary.csv')


,measure_type,sub_reason,year,value
0,asylum_inflow,asylum,2015,4331.0
1,asylum_inflow,asylum,2016,6771.0
2,asylum_inflow,asylum,2017,8444.0
3,asylum_inflow,asylum,2018,6857.0
4,asylum_inflow,asylum,2019,7896.0
5,asylum_inflow,asylum,2020,4630.0
6,asylum_inflow,asylum,2021,3606.0
7,seasonal_worker_inflow,seasonal_work,2015,0.0
8,seasonal_worker_inflow,seasonal_work,2016,0.0
9,seasonal_worker_inflow,seasonal_work,2017,2.0


## Q3 - Post-Arrival Trajectories

Eurostat is used as the main source for Q3. OECD is used as complementary evidence only and is not merged into Eurostat post-arrival totals.

The same neutral `period` groups are used only to summarize available years:

- `2015-2019`
- `2020-2021`
- `2022-2024`

In [75]:
eurostat_post_arrival = post_arrival[
    post_arrival['source'].str.lower() == 'eurostat'
].copy()

oecd_post_arrival = post_arrival[
    post_arrival['source'].str.lower() == 'oecd'
].copy()

post_arrival.groupby(['source', 'measure_type']).size().reset_index(name='rows')


,source,measure_type,rows
0,eurostat,citizenship_acquisition,400
1,eurostat,long_term_resident,330
2,eurostat,status_change,155
3,oecd,citizenship_acquisition,63


In [76]:
post_arrival_source_measure_summary = (
    post_arrival.groupby(['source', 'measure_type', 'year'], as_index=False)['value']
    .sum()
    .sort_values(['source', 'measure_type', 'year'])
)

export_csv(post_arrival_source_measure_summary, 'post_arrival_source_measure_summary.csv')


,source,measure_type,year,value
0,eurostat,citizenship_acquisition,2015,6638.0
1,eurostat,citizenship_acquisition,2016,7226.0
2,eurostat,citizenship_acquisition,2017,6612.0
3,eurostat,citizenship_acquisition,2018,5771.0
4,eurostat,citizenship_acquisition,2019,5942.0
5,eurostat,citizenship_acquisition,2020,4941.0
6,eurostat,citizenship_acquisition,2021,6836.0
7,eurostat,citizenship_acquisition,2022,6575.0
8,eurostat,citizenship_acquisition,2023,6025.0
9,eurostat,citizenship_acquisition,2024,6503.0


In [77]:
eurostat_post_arrival['period'] = eurostat_post_arrival['year'].apply(assign_period)
eurostat_post_arrival = eurostat_post_arrival[
    eurostat_post_arrival['period'] != 'outside_scope'
].copy()

eurostat_post_arrival['period'] = pd.Categorical(
    eurostat_post_arrival['period'],
    categories=period_order,
    ordered=True,
)

post_arrival_totals = export_csv(
    eurostat_post_arrival
    .groupby('measure_type', as_index=False)['value']
    .sum()
    .sort_values('value', ascending=False),
    'post_arrival_total_by_measure_type.csv',
)

post_arrival_yearly = export_csv(
    eurostat_post_arrival
    .groupby(['year', 'measure_type'], as_index=False)['value']
    .sum()
    .sort_values(['year', 'measure_type']),
    'post_arrival_yearly_evolution.csv',
)

post_arrival_by_period = (
    eurostat_post_arrival
    .groupby(['period', 'measure_type'], as_index=False, observed=False)['value']
    .sum()
)

post_arrival_by_period['period'] = pd.Categorical(
    post_arrival_by_period['period'],
    categories=period_order,
    ordered=True,
)

post_arrival_by_period = export_csv(
    post_arrival_by_period
    .sort_values(['period', 'measure_type']),
    'post_arrival_by_period.csv',
)

post_arrival_totals


,measure_type,value
1,long_term_resident,569803.0
0,citizenship_acquisition,63069.0
2,status_change,24728.0


In [78]:
top_countries_by_measure = (
    eurostat_post_arrival.groupby(['measure_type', 'destination_country'], as_index=False)['value']
    .sum()
    .sort_values(['measure_type', 'value'], ascending=[True, False])
)

top_10_post_arrival_countries_by_measure = export_csv(
    top_countries_by_measure.groupby('measure_type').head(10).reset_index(drop=True),
    'top_10_post_arrival_countries_by_measure.csv',
)

top_10_post_arrival_countries_by_measure.head(15)


,measure_type,destination_country,value
0,citizenship_acquisition,France,27689.0
1,citizenship_acquisition,Belgium,10628.0
2,citizenship_acquisition,Germany,9694.0
3,citizenship_acquisition,Italy,5011.0
4,citizenship_acquisition,United Kingdom,2343.0
5,citizenship_acquisition,Spain,2244.0
6,citizenship_acquisition,Switzerland,1614.0
7,citizenship_acquisition,Sweden,973.0
8,citizenship_acquisition,Netherlands,718.0
9,citizenship_acquisition,Finland,559.0


In [79]:
long_term_residents = eurostat_post_arrival[
    eurostat_post_arrival['measure_type'].str.lower() == 'long_term_resident'
].copy()

long_term_yearly = export_csv(
    long_term_residents.groupby('year', as_index=False)['value'].sum().sort_values('year'),
    'long_term_residents_yearly.csv',
)

long_term_by_country = export_csv(
    long_term_residents.groupby('destination_country', as_index=False)['value'].sum().sort_values('value', ascending=False),
    'long_term_residents_by_country.csv',
)

long_term_yearly


,year,value
0,2015,47577.0
1,2016,54621.0
2,2017,56946.0
3,2018,55529.0
4,2019,51678.0
5,2020,54959.0
6,2021,55627.0
7,2022,61990.0
8,2023,64287.0
9,2024,66589.0


In [80]:
status_changes = eurostat_post_arrival[
    eurostat_post_arrival['measure_type'].str.lower() == 'status_change'
].copy()

status_changes_yearly = export_csv(
    status_changes.groupby('year', as_index=False)['value'].sum().sort_values('year'),
    'status_changes_yearly.csv',
)

status_changes_by_country = export_csv(
    status_changes.groupby('destination_country', as_index=False)['value'].sum().sort_values('value', ascending=False),
    'status_changes_by_country.csv',
)

status_changes_yearly


,year,value
0,2020,2471.0
1,2021,4258.0
2,2022,5166.0
3,2023,6131.0
4,2024,6702.0


In [81]:
citizenship_acquisition = eurostat_post_arrival[
    eurostat_post_arrival['measure_type'].str.lower() == 'citizenship_acquisition'
].copy()

citizenship_yearly = export_csv(
    citizenship_acquisition.groupby('year', as_index=False)['value'].sum().sort_values('year'),
    'citizenship_acquisition_yearly.csv',
)

citizenship_by_country = export_csv(
    citizenship_acquisition.groupby('destination_country', as_index=False)['value'].sum().sort_values('value', ascending=False),
    'citizenship_acquisition_by_country.csv',
)

citizenship_yearly


,year,value
0,2015,6638.0
1,2016,7226.0
2,2017,6612.0
3,2018,5771.0
4,2019,5942.0
5,2020,4941.0
6,2021,6836.0
7,2022,6575.0
8,2023,6025.0
9,2024,6503.0


### OECD Complementary Post-Arrival Indicators

OECD post-arrival indicators are useful as supporting evidence, but they are not merged with Eurostat Q3 totals because source coverage and definitions may differ.


In [82]:
oecd_post_arrival_indicators_summary = (
    oecd_post_arrival.groupby(['measure_type', 'year'], as_index=False)['value']
    .sum()
    .sort_values(['measure_type', 'year'])
)

export_csv(oecd_post_arrival_indicators_summary, 'oecd_post_arrival_indicators_summary.csv')


,measure_type,year,value
0,citizenship_acquisition,2015,5677.0
1,citizenship_acquisition,2016,6095.0
2,citizenship_acquisition,2017,5722.0
3,citizenship_acquisition,2018,4892.0
4,citizenship_acquisition,2019,5082.0
5,citizenship_acquisition,2020,4412.0
6,citizenship_acquisition,2021,6304.0


## Methodological Caution

- Stocks are not added to flows.
- Permits are not added to citizenship acquisitions.
- OECD indicators are complementary evidence only.
- Each `measure_type` must be interpreted according to its own definition.
- The `period` variable is used as a descriptive grouping of available years.
- The notebook does not estimate causal effects linked to external events.
